# VAYU — Predictive Modeling and Unsupervised Clustering of Ambient Air Quality Across Indian Urban Centres

**Dataset:** CPCB (Central Pollution Control Board) Sensor Data  
**Pipeline:** Data Cleaning → Regression → Classification → Clustering → Dimensionality Reduction

## Problem Statement

Air pollution is one of the most critical public health challenges in India. 
The Central Pollution Control Board (CPCB) operates a nationwide network of 
Continuous Ambient Air Quality Monitoring Stations (CAAQMS) that record 
concentrations of six key pollutants — PM2.5, PM10, NO2, SO2, CO, and O3 — 
on an hourly basis across Indian cities.

Despite the scale of this data collection effort, two fundamental challenges remain:

**1. Prediction (Supervised Learning)**  
Given the current pollutant concentrations and time context, can we accurately 
predict how bad the air quality is — either as a precise number (AQI) or as a 
human-interpretable category (Good / Moderate / Severe)?

**2. Pattern Discovery (Unsupervised Learning)**  
Without any labels, can we discover natural groupings among Indian cities based 
on their long-term pollution signatures — identifying which cities share similar 
chronic pollution patterns, seasonal behaviour, and dominant pollutants?

---

### Formal Problem Definitions

| Task | Type | Input | Output |
|---|---|---|---|
| **Task 1** | Regression | PM2.5, PM10, NO2, SO2, CO, O3 + time features | AQI as a continuous number (0–500) |
| **Task 2** | Classification | Same pollutant + time features + city | AQI category label: Good / Satisfactory / Moderate / Poor / Very Poor / Severe |
| **Task 3** | Clustering | City-level aggregated AQI profile (mean, std, seasonal means, pollutant dominance) | Cluster assignment — which cities are pollution-similar? |
| **Task 4** | Dimensionality Reduction | 6-dimensional pollutant matrix (842k rows) | 2D representation preserving pollutant co-occurrence structure |

---

### Why This Problem Matters

- India has **14 of the world's 20 most polluted cities** (IQAir 2023)
- CPCB's AQI threshold of 200 (Moderate) triggers public health advisories
- Predicting AQI from raw sensor readings enables **early warning systems**
- Clustering cities by pollution signature enables **targeted policy interventions** — 
  PM2.5-heavy northern cities need different solutions than ozone-heavy southern cities

---

### Scope of This Notebook

This notebook covers **Step 1 of the pipeline: Data Understanding and Cleaning**.

Before any model can be trained, the raw CPCB data must be:
1. Audited for structural issues (wrong dtypes, sentinel values, impossible readings)
2. Cleaned systematically (7 operations in defined order)
3. Validated — distributions before vs after cleaning must be compared

The output of this notebook is `master_cleaned.csv` — the single source of truth 
that feeds all four ML tasks above.

## Dataset

### Primary Source — `aqi_india_38cols_knn_final.csv`

| Property | Value |
|---|---|
| Rows | 842,160 |
| Cities | 29 Indian cities |
| Time range | 2015 – 2024 (hourly) |
| Pollutants | PM2.5, PM10, NO2, SO2, CO, O3 |
| Pre-processing | KNN-imputed — 0% standard NaN |
| Known issue | Sentinel value **999** used by CPCB to flag sensor errors |

### Secondary Source — 277 `*_AQIBulletins.csv` files

| Property | Value |
|---|---|
| Files | 277 (one per city) |
| Cities covered | 277 Indian cities |
| Data type | Daily aggregated AQI only (no raw pollutant readings) |
| Used for | K-Means clustering task only |

### Why These Sources Were Chosen

The primary file was selected from a scan of **299 raw files** because it is the only 
file that combines all of: 842k rows of depth, all 6 CPCB pollutants, 9 years of 
temporal coverage, and 29 cities in a single consistent schema.

The bulletin files were chosen for clustering because they cover **277 cities** — 
giving far richer geographic spread than the 29-city primary file.

## Data Access

This notebook pulls data directly from two HuggingFace dataset repositories:

| Repository | Contents | Path after download |
|---|---|---|
| `rachitgoyell/vayu-raw` | 299 raw CPCB CSV files | `./data/` |
| `rachitgoyell/vayu-cleaned` | Cleaned, model-ready outputs | `./data/cleaned/` |

The download cell below sets up the exact folder structure the pipeline expects.  
No manual downloading or path changes are needed — just run the cell.

> **Note:** Both repositories are public. No HuggingFace token is required.

In [ ]:
# ── HuggingFace Dataset Setup ────────────────────────────────────────────────
import os
from huggingface_hub import snapshot_download

HF_USERNAME = "rachitgoyell"

# Download raw data → ./data/
snapshot_download(
    repo_id   = f"{HF_USERNAME}/vayu-raw",
    repo_type = "dataset",
    local_dir = "./data"
)

# Download cleaned data → ./data/cleaned/
snapshot_download(
    repo_id   = f"{HF_USERNAME}/vayu-cleaned",
    repo_type = "dataset",
    local_dir = "./data/cleaned"
)

DATA_ROOT   = './data'
OUTPUT_ROOT = './data/cleaned'

print("✓ data/         — raw files ready")
print("✓ data/cleaned/ — cleaned files ready")